# Refactoring Tasks
> Utility extraction, import lifting, and tutorial validation

In [ ]:
#| default_exp tasks.refactor

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional, Set, Tuple
from dataclasses import dataclass, field, asdict
from pathlib import Path
from collections import defaultdict
import ast
import re

from mcp.server.fastmcp import FastMCP

from nbdev_mcp.utils.paths import (
    resolve_selector, iter_notebooks, read_nb, nbs_dir, settings_dict
)
from nbdev_mcp.utils.nb import join_source, find_default_exp, extract_imports

## Import Lifting

In [ ]:
#| export
@dataclass
class ScopedImport:
    """An import found inside a function or class.
    
    Attributes
    ----------
    notebook : str
        Notebook path.
    cell_index : int
        Cell index.
    scope_name : str
        Function or class name containing the import.
    scope_type : str
        'function' or 'class'.
    import_statement : str
        The import statement.
    module : str
        Module being imported.
    line_number : int
        Line number within cell.
    """
    notebook: str
    cell_index: int
    scope_name: str
    scope_type: str
    import_statement: str
    module: str
    line_number: int
    
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

In [ ]:
#| export
def find_scoped_imports(nb_path: Path, project: Path) -> List[ScopedImport]:
    """Find imports inside functions and classes.
    
    Parameters
    ----------
    nb_path : Path
        Path to notebook.
    project : Path
        Project root.
    
    Returns
    -------
    List[ScopedImport]
        Scoped imports found.
    """
    imports = []
    nb_name = str(nb_path.relative_to(project))
    
    try:
        nb_data = read_nb(nb_path)
    except Exception:
        return imports
    
    for cell_idx, cell in enumerate(nb_data.get('cells', [])):
        if cell.get('cell_type') != 'code':
            continue
        
        src = join_source(cell.get('source', []))
        
        try:
            tree = ast.parse(src)
        except SyntaxError:
            continue
        
        # Walk the AST looking for imports inside functions/classes
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                scope_name = node.name
                scope_type = 'class' if isinstance(node, ast.ClassDef) else 'function'
                
                for child in ast.walk(node):
                    if isinstance(child, ast.Import):
                        for alias in child.names:
                            imports.append(ScopedImport(
                                notebook=nb_name,
                                cell_index=cell_idx,
                                scope_name=scope_name,
                                scope_type=scope_type,
                                import_statement=f"import {alias.name}",
                                module=alias.name,
                                line_number=child.lineno
                            ))
                    elif isinstance(child, ast.ImportFrom):
                        if child.module:
                            names = ', '.join(a.name for a in child.names)
                            imports.append(ScopedImport(
                                notebook=nb_name,
                                cell_index=cell_idx,
                                scope_name=scope_name,
                                scope_type=scope_type,
                                import_statement=f"from {child.module} import {names}",
                                module=child.module,
                                line_number=child.lineno
                            ))
    
    return imports


def find_all_scoped_imports(
    project: Optional[str] = None
) -> Dict[str, Any]:
    """Find all scoped imports across the project.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    
    Returns
    -------
    Dict[str, Any]
        Grouped scoped imports.
    """
    p = resolve_selector(project)
    all_imports = []
    
    for nb_path in iter_notebooks(p):
        imports = find_scoped_imports(nb_path, p)
        all_imports.extend(imports)
    
    # Group by notebook
    by_notebook: Dict[str, List[Dict]] = defaultdict(list)
    for imp in all_imports:
        by_notebook[imp.notebook].append(imp.to_dict())
    
    return {
        'ok': True,
        'total_scoped_imports': len(all_imports),
        'by_notebook': dict(by_notebook),
        'recommendation': 'Move these imports to module level for cleaner code'
    }

## Tutorial Validation

In [ ]:
#| export
@dataclass
class TutorialIssue:
    """An issue found in a tutorial notebook.
    
    Attributes
    ----------
    notebook : str
        Tutorial notebook path.
    cell_index : int
        Cell index with issue.
    issue_type : str
        Type of issue.
    message : str
        Issue description.
    traceback : str, optional
        Error traceback if execution failed.
    """
    notebook: str
    cell_index: int
    issue_type: str
    message: str
    traceback: Optional[str] = None
    
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

In [ ]:
#| export
def validate_tutorial_structure(
    nb_path: Path,
    project: Path,
    lib_name: str
) -> List[TutorialIssue]:
    """Validate a tutorial notebook's structure.
    
    Checks:
    - Has title markdown cell
    - Imports the library correctly
    - No export directives (tutorials shouldn't export)
    - Has runnable examples
    
    Parameters
    ----------
    nb_path : Path
        Tutorial notebook path.
    project : Path
        Project root.
    lib_name : str
        Library name.
    
    Returns
    -------
    List[TutorialIssue]
        Issues found.
    """
    issues = []
    nb_name = str(nb_path.relative_to(project))
    
    try:
        nb_data = read_nb(nb_path)
    except Exception as e:
        issues.append(TutorialIssue(
            notebook=nb_name,
            cell_index=0,
            issue_type='parse_error',
            message=f'Failed to parse notebook: {e}'
        ))
        return issues
    
    cells = nb_data.get('cells', [])
    
    # Check for title
    has_title = False
    for i, cell in enumerate(cells):
        if cell.get('cell_type') == 'markdown':
            src = join_source(cell.get('source', []))
            if src.strip().startswith('#'):
                has_title = True
                break
    
    if not has_title:
        issues.append(TutorialIssue(
            notebook=nb_name,
            cell_index=0,
            issue_type='missing_title',
            message='Tutorial should start with a markdown title'
        ))
    
    # Check for exports (bad in tutorials)
    lib_name_underscore = lib_name.replace('-', '_')
    has_library_import = False
    
    for i, cell in enumerate(cells):
        if cell.get('cell_type') != 'code':
            continue
        
        src = join_source(cell.get('source', []))
        
        # Check for export directives
        if '#| export' in src or '#|export' in src:
            issues.append(TutorialIssue(
                notebook=nb_name,
                cell_index=i,
                issue_type='has_export',
                message='Tutorials should not have export directives'
            ))
        
        # Check for default_exp
        if '#| default_exp' in src or '#|default_exp' in src:
            issues.append(TutorialIssue(
                notebook=nb_name,
                cell_index=i,
                issue_type='has_default_exp',
                message='Tutorials should not have default_exp directive'
            ))
        
        # Check for library import
        if f'import {lib_name_underscore}' in src or f'from {lib_name_underscore}' in src:
            has_library_import = True
    
    if not has_library_import:
        issues.append(TutorialIssue(
            notebook=nb_name,
            cell_index=0,
            issue_type='no_library_import',
            message=f'Tutorial should import {lib_name_underscore}'
        ))
    
    return issues


def scan_tutorial_errors(
    nb_path: Path,
    project: Path
) -> List[TutorialIssue]:
    """Scan a tutorial notebook for existing error outputs.
    
    Parameters
    ----------
    nb_path : Path
        Tutorial notebook path.
    project : Path
        Project root.
    
    Returns
    -------
    List[TutorialIssue]
        Error issues found.
    """
    issues = []
    nb_name = str(nb_path.relative_to(project))
    
    try:
        nb_data = read_nb(nb_path)
    except Exception:
        return issues
    
    for i, cell in enumerate(nb_data.get('cells', [])):
        if cell.get('cell_type') != 'code':
            continue
        
        outputs = cell.get('outputs', [])
        for output in outputs:
            if output.get('output_type') == 'error':
                ename = output.get('ename', 'Error')
                evalue = output.get('evalue', '')
                traceback_lines = output.get('traceback', [])
                
                issues.append(TutorialIssue(
                    notebook=nb_name,
                    cell_index=i,
                    issue_type='error_output',
                    message=f'{ename}: {evalue}',
                    traceback='\n'.join(traceback_lines)[:1000]  # Limit size
                ))
    
    return issues

In [ ]:
#| export
def validate_all_tutorials(
    project: Optional[str] = None,
    check_errors: bool = True
) -> Dict[str, Any]:
    """Validate all tutorial notebooks.
    
    Looks for tutorials in tutorials/ directory.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    check_errors : bool
        Whether to scan for error outputs.
    
    Returns
    -------
    Dict[str, Any]
        Validation results.
    """
    p = resolve_selector(project)
    settings = settings_dict(p)
    lib_name = settings.get('lib_name', 'pkg')
    
    # Find tutorials directory
    tutorials_dir = p / 'tutorials'
    if not tutorials_dir.exists():
        nbs = nbs_dir(p)
        tutorials_dir = nbs / 'tutorials'
    
    if not tutorials_dir.exists():
        return {
            'ok': True,
            'message': 'No tutorials directory found',
            'tutorials': []
        }
    
    all_issues = []
    tutorial_count = 0
    
    for nb_path in tutorials_dir.glob('*.ipynb'):
        if nb_path.name.startswith('.'):
            continue
        
        tutorial_count += 1
        
        # Structure validation
        issues = validate_tutorial_structure(nb_path, p, lib_name)
        all_issues.extend(issues)
        
        # Error scanning
        if check_errors:
            error_issues = scan_tutorial_errors(nb_path, p)
            all_issues.extend(error_issues)
    
    # Group by notebook
    by_notebook: Dict[str, List[Dict]] = defaultdict(list)
    for issue in all_issues:
        by_notebook[issue.notebook].append(issue.to_dict())
    
    return {
        'ok': len(all_issues) == 0,
        'tutorial_count': tutorial_count,
        'total_issues': len(all_issues),
        'by_notebook': dict(by_notebook)
    }

## Code Pattern Detection

In [ ]:
#| export
@dataclass
class PatternMatch:
    """A code pattern match.
    
    Attributes
    ----------
    notebook : str
        Notebook path.
    cell_index : int
        Cell index.
    line_start : int
        Start line.
    line_end : int
        End line.
    matched_text : str
        The matched code.
    context : str
        Surrounding context.
    """
    notebook: str
    cell_index: int
    line_start: int
    line_end: int
    matched_text: str
    context: str
    
    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

In [ ]:
#| export
def find_pattern(
    project: Optional[str] = None,
    pattern: str = '',
    in_exports_only: bool = True
) -> Dict[str, Any]:
    """Find code matching a regex pattern.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    pattern : str
        Regex pattern to match.
    in_exports_only : bool
        Only search in export cells.
    
    Returns
    -------
    Dict[str, Any]
        Pattern matches.
    """
    p = resolve_selector(project)
    matches = []
    
    try:
        regex = re.compile(pattern, re.MULTILINE)
    except re.error as e:
        return {'ok': False, 'error': f'Invalid regex: {e}'}
    
    for nb_path in iter_notebooks(p):
        nb_name = str(nb_path.relative_to(p))
        
        try:
            nb_data = read_nb(nb_path)
        except Exception:
            continue
        
        for cell_idx, cell in enumerate(nb_data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            
            src = join_source(cell.get('source', []))
            
            # Skip non-export cells if requested
            if in_exports_only:
                if '#| export' not in src and '#|export' not in src:
                    continue
            
            for match in regex.finditer(src):
                # Calculate line numbers
                lines_before = src[:match.start()].count('\n')
                matched_lines = match.group().count('\n')
                
                # Get context (surrounding lines)
                all_lines = src.split('\n')
                ctx_start = max(0, lines_before - 2)
                ctx_end = min(len(all_lines), lines_before + matched_lines + 3)
                context = '\n'.join(all_lines[ctx_start:ctx_end])
                
                matches.append(PatternMatch(
                    notebook=nb_name,
                    cell_index=cell_idx,
                    line_start=lines_before + 1,
                    line_end=lines_before + matched_lines + 1,
                    matched_text=match.group()[:200],  # Limit size
                    context=context[:500]  # Limit size
                ))
    
    return {
        'ok': True,
        'pattern': pattern,
        'total_matches': len(matches),
        'matches': [m.to_dict() for m in matches]
    }

## Extraction Suggestions

In [ ]:
#| export
def suggest_extractions(
    project: Optional[str] = None,
    min_occurrences: int = 3,
    min_lines: int = 3
) -> Dict[str, Any]:
    """Suggest code blocks that could be extracted as utilities.
    
    Looks for repeated code patterns that appear multiple times.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    min_occurrences : int
        Minimum number of occurrences.
    min_lines : int
        Minimum lines of code.
    
    Returns
    -------
    Dict[str, Any]
        Extraction suggestions.
    """
    p = resolve_selector(project)
    
    # Collect all code blocks (multi-line statements)
    code_blocks: Dict[str, List[Tuple[str, int]]] = defaultdict(list)  # hash -> [(notebook, cell)]
    
    for nb_path in iter_notebooks(p):
        nb_name = str(nb_path.relative_to(p))
        
        try:
            nb_data = read_nb(nb_path)
        except Exception:
            continue
        
        for cell_idx, cell in enumerate(nb_data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            
            src = join_source(cell.get('source', []))
            
            # Only check export cells
            if '#| export' not in src and '#|export' not in src:
                continue
            
            try:
                tree = ast.parse(src)
            except SyntaxError:
                continue
            
            # Look at each statement
            for node in ast.iter_child_nodes(tree):
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                    # Look inside functions for repeated patterns
                    for child in ast.iter_child_nodes(node):
                        if hasattr(child, 'lineno') and hasattr(child, 'end_lineno'):
                            if child.end_lineno - child.lineno + 1 >= min_lines:
                                try:
                                    block_src = ast.get_source_segment(src, child)
                                    if block_src:
                                        # Normalize for comparison
                                        normalized = ast.dump(ast.parse(block_src))
                                        import hashlib
                                        block_hash = hashlib.md5(normalized.encode()).hexdigest()[:12]
                                        code_blocks[block_hash].append((nb_name, cell_idx, block_src))
                                except:
                                    pass
    
    # Find repeated blocks
    suggestions = []
    for block_hash, occurrences in code_blocks.items():
        if len(occurrences) >= min_occurrences:
            sample = occurrences[0][2][:300] if len(occurrences[0]) > 2 else ''
            suggestions.append({
                'hash': block_hash,
                'occurrences': len(occurrences),
                'locations': [(nb, cell) for nb, cell, _ in occurrences],
                'sample': sample,
                'suggestion': 'Extract as utility function'
            })
    
    # Sort by occurrences
    suggestions.sort(key=lambda x: x['occurrences'], reverse=True)
    
    return {
        'ok': True,
        'total_suggestions': len(suggestions),
        'suggestions': suggestions[:20]  # Limit to top 20
    }

## MCP Tool Registration

In [ ]:
#| export
def add_refactor_tools(mcp: FastMCP) -> None:
    """Register refactoring tools with MCP server.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance.
    """
    
    @mcp.tool()
    def lift_imports(
        project: Optional[str] = None
    ) -> Dict[str, Any]:
        """Find imports inside functions/classes that should be lifted.
        
        Scoped imports are usually a code smell - they should be at module level.
        """
        return find_all_scoped_imports(project)
    
    @mcp.tool()
    def validate_tutorials(
        project: Optional[str] = None,
        check_errors: bool = True
    ) -> Dict[str, Any]:
        """Validate tutorial notebooks for common issues.
        
        Checks structure, imports, and error outputs.
        """
        return validate_all_tutorials(project, check_errors)
    
    @mcp.tool()
    def search_pattern(
        project: Optional[str] = None,
        pattern: str = '',
        exports_only: bool = True
    ) -> Dict[str, Any]:
        """Search for a regex pattern across notebooks.
        
        Useful for finding specific code patterns to refactor.
        """
        return find_pattern(project, pattern, exports_only)
    
    @mcp.tool()
    def extraction_suggestions(
        project: Optional[str] = None,
        min_occurrences: int = 3
    ) -> Dict[str, Any]:
        """Suggest code blocks that could be extracted as utilities.
        
        Finds repeated code patterns across notebooks.
        """
        return suggest_extractions(project, min_occurrences)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()